In [ ]:
from pathlib import Path
from typing import Literal
import pickle
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm

import python.utils as utils
import python.plots as plots
from python.error_metrics import MergedErrorMetrics, LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.forecast_reconciliation import reconcile_calculate_serialize_error_metrics
from python.sampled_hierarchies import SampledHierarchies
from python.statistical_test import TrendType, l_test_stats, t_test_stats, TestResultsLatex
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS
from python.aggregation import MONTHS, DAYS
from python.constants import AVAILABLE_LEVELS
from python.base_forecasting import MODEL_MIN_TRAIN_OBS

# Settings

In [ ]:
DATASET: Literal['web-traffic', 'M5', 'M4'] = 'M5'
MODEL: Literal['AutoETS', 'AutoARIMA', 'AutoTheta', 'Prophet'] = 'AutoARIMA'
HORIZON: int = 3 * 28 # DAYS
START, END = 16000, 31000 # To trim dataset
N_JOBS: int = 2
VERBOSE: bool = True
ENVIRONMENT: str = 'cpu'


FOLDER = f"{DATASET}/{MODEL}"
FORECAST_CHECKPOINT = Path(f'sample/{FOLDER}/all_possible-{START}-{END}.pkl')
CAPTION_SUFFIX = f"{DATASET} dataset - {MODEL} model"
LABEL_SUFFIX = f"{DATASET}_{MODEL}"

Path(f"sample/{FOLDER}").mkdir(parents=True, exist_ok=True)
Path(f"metrics/{FOLDER}").mkdir(parents=True, exist_ok=True)

MIN_SERIES_LENGTH = HORIZON + MODEL_MIN_TRAIN_OBS[MODEL] * 28

os.environ["JAX_PLATFORMS"] = ENVIRONMENT

# Forecasts and errors

In [ ]:
if DATASET == 'web-traffic':
    X_raw = utils.fetch_dataset_web_traffic()
    X = utils.filter_last_days_available(X_raw, 28*100)
if DATASET == 'M5':
    X = utils.fetch_dataset_m5()
if DATASET == 'M4':
    X = utils.fetch_dataset_m4()

X = utils.filter_last_days_available(X, HORIZON)
X = utils.reduce_dataset(X, start=START, end=END)
X = utils.align_fill_and_trim_series(X)
X = utils.filter_min_length(X, MIN_SERIES_LENGTH)

In [ ]:
sh = SampledHierarchies(AVAILABLE_LEVELS, 3, 10)
all_possible = sh.full_hierarchy
minimal = sh.base_hierarchy

In [ ]:
all_possible.resample_time_series(X)

for cp in all_possible.make_base_forecasts(HORIZON, model=MODEL, n_jobs=N_JOBS, verbose=VERBOSE):
    with open(FORECAST_CHECKPOINT, 'wb') as f:
        pickle.dump(cp, f)

In [ ]:
with open(FORECAST_CHECKPOINT, 'rb') as f:
    all_possible = pickle.load(f)

for h in sh.hierarchies + [minimal]:
    h.copy_data(all_possible)

In [ ]:
reconcile_calculate_serialize_error_metrics(sh.hierarchies + [minimal],
                                            lambda h: f'sample/{FOLDER}/metrics-{h.name}-{START}-{END}.csv',
                                            verbose=VERBOSE,
                                            n_jobs=N_JOBS)

In [ ]:
from pathlib import Path
ems = []
for file in Path(f"sample/{FOLDER}").rglob(f"*-{START}-{END}.csv"):
    if file.is_file():
        ems.append(LoadedErrorMetrics(file))

em = MergedErrorMetrics(ems)

em.save(f'metrics/{FOLDER}/sample_hiers_errors_raw-{START}-{END}.csv')

In [ ]:
em = LoadedErrorMetrics(f'metrics/{FOLDER}/sample_hiers_errors_raw-{START}-{END}.csv')

em = MergedErrorMetrics([
    FilteredErrorMetrics(em, MI(forecast_type=RECONCILED_FORECASTS)),
    FilteredErrorMetrics(em, MI(forecast_type=BASE_FORECASTS, hierarchy='Minimal'))
])

em.save(f'metrics/{FOLDER}/sample_hiers_errors.csv')

# Calculations

In [ ]:
ERROR_METRIC = 'nrmse'
STATISTIC = 'winsorized'

In [ ]:
sh = SampledHierarchies(AVAILABLE_LEVELS, 3, 10)
all_possible = sh.full_hierarchy
minimal = sh.base_hierarchy

em = FilteredErrorMetrics(
    LoadedErrorMetrics(f'metrics/{FOLDER}/sample_hiers_errors.csv'),
    MI(error_metric=ERROR_METRIC)
)

In [ ]:
mon_rec_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS))
day_rec_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=DAYS,   forecast_type=RECONCILED_FORECASTS))

base_stats = em.get_error_stats(MI(hierarchy=minimal, forecast_type=BASE_FORECASTS))

In [ ]:
plots.latex_stats_table(base_stats, STATISTIC, 'granularity',
                        stat_label='Winsorized Mean NRMSE',
                        caption=fr'Baseline base forecast error statistics (winsorized mean NRMSE at 0.1\%) - {CAPTION_SUFFIX}',
                        label=f'baseline_{LABEL_SUFFIX}')

In [ ]:
plots.latex_sampled_error_table(sh, STATISTIC, stats_matrix=mon_rec_stats_matrix,
                                caption=fr'Winsorized mean NRMSE (0.1\%) at monthly granularity per hierarchy - {CAPTION_SUFFIX}',
                                label=f'month_mean_{LABEL_SUFFIX}')

In [ ]:
plots.latex_sampled_error_table(sh, STATISTIC, stats_matrix=day_rec_stats_matrix,
                                caption=fr'Winsorized mean NRMSE (0.1\%) at daily granularity per hierarchy - {CAPTION_SUFFIX}',
                                label=f'day_mean_{LABEL_SUFFIX}')

In [ ]:
plots.latex_conformism_table(sh,
                             {"Monthly (\\%)": mon_rec_stats_matrix,
                              "Daily (\\%)": day_rec_stats_matrix},
                              STATISTIC,
                              'decrease',
                              caption=f'Theorem conformism percentage per sequence - {CAPTION_SUFFIX}',
                              label=f'conformism_{LABEL_SUFFIX}')

In [ ]:
TestResultsLatex(
    l_test_stats(mon_rec_stats_matrix, STATISTIC, TrendType.DECREASING),
    t_test_stats(mon_rec_stats_matrix, STATISTIC, TrendType.DECREASING),
).to_table(title=f'Statistical test results for decreasing trends in monthly winsorized mean NRMSE - {CAPTION_SUFFIX}',
           star=False,
           label=f'monthly_test_{LABEL_SUFFIX}')

In [ ]:
TestResultsLatex(
    l_test_stats(day_rec_stats_matrix, STATISTIC, TrendType.DECREASING),
    t_test_stats(day_rec_stats_matrix, STATISTIC, TrendType.DECREASING),
).to_table(title=f'Statistical test results for decreasing trends in daily winsorized mean NRMSE - {CAPTION_SUFFIX}',
           star=False,
           label=f'monthly_test_{LABEL_SUFFIX}')